In [0]:
-- Definition of done
-- ok Quantidade de transações históricas (vida, D7, D14, D28, D56);
-- ok Dias desde a última transação -- recencia
-- ok Dias desde a primeira transação
-- ok Saldo de pontos atual;
-- ok Pontos acumulados positivos (vida, D7, D14, D28, D56);
-- ok Pontos acumulados negativos (vida, D7, D14, D28, D56);
-- Dias da semana mais ativos (D28)
-- Período do dia mais ativo (D28)
-- Engajamento em D28 versus Vida -> qtdTransacao28 / qtdTransacaoVida
-- Produto mais usado (vida, D7, D14, D28, D56);

WITH tb_stats AS (
    SELECT IdCliente,
          count(IdTransacao) as qtdTransacaoVida,
          count(CASE WHEN date_diff('2026-01-01', DtCriacao) <= 7 THEN IdTransacao END) AS qtdeTransacaoD7,
          count(CASE WHEN date_diff('2026-01-01', DtCriacao) <= 14 THEN IdTransacao END) AS qtdeTransacaoD14,
          count(CASE WHEN date_diff('2026-01-01', DtCriacao) <= 28 THEN IdTransacao END) AS qtdeTransacaoD28,
          count(CASE WHEN date_diff('2026-01-01', DtCriacao) <= 56 THEN IdTransacao END) AS qtdeTransacaoD56,
          max(DtCriacao) AS dtUltimaCompra,
          min(date_diff('2026-01-01', DtCriacao)) AS diasUltimaCompra,
          max(date_diff('2026-01-01', DtCriacao)) AS diasPrimeiraCompra,
          sum(QtdePontos) AS saldoPontos,

          sum(CASE WHEN QtdePontos > 0 THEN QtdePontos END) AS qtdePontosPosVida,
          sum(CASE WHEN date_diff('2026-01-01', DtCriacao) <= 7 AND QtdePontos > 0 THEN QtdePontos ELSE 0 END) AS qtdePontosPos7,
          sum(CASE WHEN date_diff('2026-01-01', DtCriacao) <= 14 AND QtdePontos > 0 THEN QtdePontos ELSE 0 END) AS qtdePontosPos14,
          sum(CASE WHEN date_diff('2026-01-01', DtCriacao) <= 28 AND QtdePontos > 0 THEN QtdePontos ELSE 0 END) AS qtdePontosPos28,
          sum(CASE WHEN date_diff('2026-01-01', DtCriacao) <= 56 AND QtdePontos > 0 THEN QtdePontos ELSE 0 END) AS qtdePontosPos56,

          sum(CASE WHEN QtdePontos < 0 THEN QtdePontos ELSE 0 END) AS qtdePontosNegVida,
          sum(CASE WHEN date_diff('2026-01-01', DtCriacao) <= 7 AND QtdePontos < 0 THEN QtdePontos ELSE 0 END) AS qtdePontosNeg7,
          sum(CASE WHEN date_diff('2026-01-01', DtCriacao) <= 14 AND QtdePontos < 0 THEN QtdePontos ELSE 0 END) AS qtdePontosNeg14,
          sum(CASE WHEN date_diff('2026-01-01', DtCriacao) <= 28 AND QtdePontos < 0 THEN QtdePontos ELSE 0 END) AS qtdePontosNeg28,
          sum(CASE WHEN date_diff('2026-01-01', DtCriacao) <= 56 AND QtdePontos < 0 THEN QtdePontos ELSE 0 END) AS qtdePontosNeg56

    FROM workspace.tmw_points.transacoes
    WHERE  DtCriacao < '2026-01-01'
    GROUP BY IdCliente
),

tb_cliente_produto AS (

    -- Produto mais usado
    SELECT 
          t1.IdCliente,
          t3.DescNomeProduto,
          count(t1.IdTransacao) AS qtdeTransacao,
          count(CASE WHEN date_diff('2026-01-01', t1.DtCriacao) <= 7 THEN t1.IdTransacao END) AS qtdeTransacao7,
          count(CASE WHEN date_diff('2026-01-01', t1.DtCriacao) <= 14 THEN t1.IdTransacao END) AS qtdeTransacao14,
          count(CASE WHEN date_diff('2026-01-01', t1.DtCriacao) <= 28 THEN t1.IdTransacao END) AS qtdeTransacao28,
          count(CASE WHEN date_diff('2026-01-01', t1.DtCriacao) <= 56 THEN t1.IdTransacao END) AS qtdeTransacao56

    FROM workspace.tmw_points.transacoes as t1

    LEFT JOIN workspace.tmw_points.transacao_produto As t2
    ON t1.IdTransacao = t2.IdTransacao

    LEFT JOIN workspace.tmw_points.produtos AS t3
    ON t2.idProduto = t3.IdProduto

    WHERE  t1.DtCriacao < '2026-01-01'
    GROUP BY ALL

),

tb_rank_cliente_produto AS (

  SELECT *,
        row_number() OVER (PARTITION BY IdCliente order by qtdeTransacao DESC) as topProdutoVida,
        row_number() OVER (PARTITION BY IdCliente order by qtdeTransacao7 DESC) as topProduto7,
        row_number() OVER (PARTITION BY IdCliente order by qtdeTransacao14 DESC) as topProduto14,
        row_number() OVER (PARTITION BY IdCliente order by qtdeTransacao28 DESC) as topProduto28,
        row_number() OVER (PARTITION BY IdCliente order by qtdeTransacao56 DESC) as topProduto56

  FROM tb_cliente_produto

),

tb_cliente_top_produtos AS (

    SELECT idCliente,
          any_value(CASE WHEN topProdutoVida = 1 THEN DescNomeProduto END) AS ProdutoTopVida,
          any_value(CASE WHEN topProduto7 = 1 THEN DescNomeProduto END) AS ProdutoTop7D,
          any_value(CASE WHEN topProduto14 = 1 THEN DescNomeProduto END) AS ProdutoTop14D,
          any_value(CASE WHEN topProduto28 = 1 THEN DescNomeProduto END) AS ProdutoTop28D,
          any_value(CASE WHEN topProduto56 = 1 THEN DescNomeProduto END) AS ProdutoTop56D

    FROM tb_rank_cliente_produto
    GROUP BY idcliente
)

SELECT t1.*,
      t2.ProdutoTopVida,
      t2.ProdutoTop7D,
      t2.ProdutoTop14D,
      t2.ProdutoTop28D,
      t2.ProdutoTop56D

FROM tb_stats AS t1
LEFT JOIN tb_cliente_top_produtos AS t2
ON t1.idcliente = t2.idcliente